<a href="https://colab.research.google.com/github/Fu-Pei-Yin/Deep-Generative-Mode/blob/week5/%E4%BD%BF%E7%94%A8_Seq2Seq%E7%94%9F%E6%88%90%E3%80%8C%E6%9C%AA%E4%BE%86%E5%AD%B8%E7%BF%92%E8%A1%8C%E7%82%BA%E5%BA%8F%E5%88%97%E3%80%8D%E6%95%B8%E6%93%9A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
UPLOADED_FILES = {}
# 讓使用者選擇要上傳的檔案
uploaded = files.upload()

# 檢查上傳結果
for filename, content in uploaded.items():
  UPLOADED_FILES[filename] = content
  print(f"✅ 已上傳：{filename}")


KeyboardInterrupt: 

In [ ]:
from google.colab import files
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import zipfile

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
UPLOADED_FILES = {}
def upload_files_if_needed():
    """只在需要時上傳檔案，避免重複上傳"""
    global UPLOADED_FILES

    # 檢查是否已經上傳過檔案
    if UPLOADED_FILES:
        print("✅ 使用已上傳的檔案，跳過重新上傳")
        for filename in UPLOADED_FILES.keys():
            print(f"   - {filename}")
        return True

    print("📁 請上傳 OULAD 資料集檔案")
    print("   - studentInfo.csv")
    print("   - studentVle.csv")
    print("   - studentAssessment.csv")
    print("   或上傳包含這些檔案的 ZIP 壓縮檔")

    try:
        uploaded = files.upload()

        for filename, content in uploaded.items():
            UPLOADED_FILES[filename] = content
            print(f"✅ 已上傳：{filename}")

            # 如果是 ZIP 檔案，解壓縮
            if filename.endswith('.zip'):
                print(f"📦 解壓縮 {filename}...")
                with zipfile.ZipFile(filename, 'r') as zip_ref:
                    zip_ref.extractall('.')
                print("✅ 解壓縮完成")

                # 記錄解壓縮後的檔案
                for extracted_file in zip_ref.namelist():
                    if any(keyword in extracted_file for keyword in ['studentInfo', 'studentVle', 'studentAssessment']):
                        print(f"📄 找到檔案：{extracted_file}")

        return len(uploaded) > 0
    except Exception as e:
        print(f"❌ 上傳失敗：{e}")
        return False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class DataHandler:
    def __init__(self, min_weeks_needed=12):
        self.min_weeks_needed = min_weeks_needed
        self.student_scalers = {}
        self.global_scaler = StandardScaler()
        self.available_files = {}

    def find_uploaded_files(self):
        """尋找已上傳的檔案"""
        global UPLOADED_FILES

        # 先檢查解壓縮的檔案
        for file in os.listdir('.'):
            if file.endswith('.csv'):
                if 'studentInfo' in file:
                    self.available_files['studentInfo'] = file
                elif 'studentVle' in file:
                    self.available_files['studentVle'] = file
                elif 'studentAssessment' in file:
                    self.available_files['studentAssessment'] = file

        # 如果找不到，檢查直接上傳的 CSV 檔案
        if not self.available_files:
            for filename in UPLOADED_FILES.keys():
                if filename.endswith('.csv'):
                    if 'studentInfo' in filename:
                        self.available_files['studentInfo'] = filename
                    elif 'studentVle' in filename:
                        self.available_files['studentVle'] = filename
                    elif 'studentAssessment' in filename:
                        self.available_files['studentAssessment'] = filename

        print("🔍 找到的檔案：")
        for key, value in self.available_files.items():
            print(f"   - {key}: {value}")

        return len(self.available_files) >= 3

    def load_oulad_or_synthetic(self):
        """載入 OULAD 資料或創建模擬資料"""
        if self.find_uploaded_files():
            print("📊 載入上傳的 OULAD 檔案...")
            try:
                si = pd.read_csv(self.available_files['studentInfo'])
                sv = pd.read_csv(self.available_files['studentVle'])
                sa = pd.read_csv(self.available_files['studentAssessment'])
                print(f"✅ 成功載入：studentInfo({len(si)}), studentVle({len(sv)}), studentAssessment({len(sa)})")
                return si, sv, sa
            except Exception as e:
                print(f"❌ 載入檔案失敗：{e}")
                print("📝 創建模擬資料集...")

        else:
            print("📝 未找到 OULAD 檔案 - 創建模擬資料集")




    def preprocess_weekly(self, student_info, student_vle, student_assessment):
        """預處理每週資料"""
        records = []
        all_clicks = []

        for sid in tqdm(student_info['id_student'].unique(), desc="Preprocessing students"):
            s_clicks = student_vle[student_vle['id_student'] == sid].copy()
            if s_clicks.empty:
                continue

            # 按週彙總點擊資料
            s_clicks = s_clicks.groupby('date')['sum_click'].sum().reset_index()
            min_w, max_w = int(s_clicks['date'].min()), int(s_clicks['date'].max())
            if (max_w - min_w + 1) < self.min_weeks_needed:
                continue

            weeks = np.arange(min_w, max_w + 1)
            dfw = pd.DataFrame({"week": weeks})
            df_clicks = s_clicks.rename(columns={"date": "week", "sum_click": "clicks"})[['week', 'clicks']]
            df = dfw.merge(df_clicks, on='week', how='left').fillna(0)

            all_clicks.extend(df['clicks'].tolist())

            # 處理評量資料
            s_assess = student_assessment[student_assessment['id_student'] == sid]
            df['submit_cnt'] = 0
            df['avg_score_sofar'] = 0.0
            df['has_submit'] = 0

            cumulative_scores = []
            cumulative_count = 0
            cumulative_sum = 0.0

            for _, row in df.iterrows():
                week = row['week']
                # Corrected column name from 'date' to 'date_submitted'
                week_assessments = s_assess[s_assess['date_submitted'] == week]

                if not week_assessments.empty:
                    df.loc[df['week'] == week, 'submit_cnt'] = len(week_assessments)
                    df.loc[df['week'] == week, 'has_submit'] = 1

                    for _, assess in week_assessments.iterrows():
                        cumulative_sum += assess['score']
                        cumulative_count += 1
                        cumulative_scores.append(assess['score'])

                if cumulative_count > 0:
                    df.loc[df['week'] == week, 'avg_score_sofar'] = cumulative_sum / cumulative_count

            # 計算點擊量變化
            df['clicks_diff1'] = df['clicks'].diff().fillna(0)
            df['student_id'] = sid
            records.append(df)

        if not records:
            raise ValueError("No students with sufficient weeks after preprocessing.")

        # 標準化點擊資料
        all_clicks = np.array(all_clicks).reshape(-1, 1)
        self.global_scaler.fit(all_clicks)

        weekly = pd.concat(records, ignore_index=True)
        return weekly

    def create_sequences(self, weekly_df, seq_length=4, pred_length=2):
        """創建序列資料"""
        seqs = []
        features = ['clicks', 'submit_cnt', 'avg_score_sofar', 'clicks_diff1']

        for sid in weekly_df['student_id'].unique():
            sd = weekly_df[weekly_df['student_id'] == sid].sort_values('week').reset_index(drop=True)
            if len(sd) < seq_length + pred_length:
                continue

            data = sd[features].values.astype(np.float32)
            original_clicks = data[:, 0].copy()

            # 標準化點擊資料
            clicks_data = data[:, 0].reshape(-1, 1)
            try:
                scaled_clicks = self.global_scaler.transform(clicks_data).flatten()
            except Exception:
                continue

            data_scaled = data.copy()
            data_scaled[:, 0] = scaled_clicks

            n = len(sd)
            for i in range(0, n - seq_length - pred_length + 1):
                inp = data_scaled[i:i + seq_length]
                out = data_scaled[i + seq_length:i + seq_length + pred_length, 0]
                original_out = original_clicks[i + seq_length:i + seq_length + pred_length]

                seqs.append({
                    'input': inp.astype(np.float32),
                    'output': out.astype(np.float32),
                    'student_id': sid,
                    'original_output': original_out.astype(np.float32),
                    'week_index': i
                })

        print(f"📊 創建了 {len(seqs)} 個序列")
        return seqs

class SequenceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        s = self.sequences[idx]
        return torch.tensor(s['input'], dtype=torch.float32), torch.tensor(s['output'], dtype=torch.float32)

class Seq2SeqLSTM(nn.Module):
    def __init__(self, input_dim=4, enc_hidden=128, dec_hidden=128, num_layers=2, pred_len=2):
        super().__init__()
        self.input_dim = input_dim
        self.enc_hidden = enc_hidden
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        self.pred_len = pred_len

        self.encoder = nn.LSTM(input_dim, enc_hidden, num_layers=num_layers, batch_first=True, bidirectional=False)
        self.decoder = nn.LSTM(enc_hidden, dec_hidden, num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(dec_hidden, 1)

    def forward(self, x):
        _, (hidden, cell) = self.encoder(x)

        decoder_input = torch.zeros(x.size(0), self.pred_len, self.enc_hidden).to(x.device)
        decoder_output, _ = self.decoder(decoder_input, (hidden, cell))

        output = self.fc_out(decoder_output).squeeze(-1)
        return output

class Seq2SeqVAE(nn.Module):
    def __init__(self, input_dim=4, enc_hidden=128, latent_dim=32, dec_hidden=128, num_layers=2, pred_len=2):
        super().__init__()
        self.input_dim = input_dim
        self.enc_hidden = enc_hidden
        self.latent_dim = latent_dim
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        self.pred_len = pred_len

        self.encoder_lstm = nn.LSTM(input_dim, enc_hidden, num_layers=num_layers, batch_first=True, bidirectional=True)
        enc_output_dim = enc_hidden * 2

        self.fc_mu = nn.Linear(enc_output_dim, latent_dim)
        self.fc_logvar = nn.Linear(enc_output_dim, latent_dim)

        self.decoder_lstm = nn.LSTM(latent_dim, dec_hidden, num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(dec_hidden, 1)

    def encode(self, x):
        _, (hidden, _) = self.encoder_lstm(x)
        hidden_forward = hidden[-2]
        hidden_backward = hidden[-1]
        hidden_concat = torch.cat([hidden_forward, hidden_backward], dim=1)

        mu = self.fc_mu(hidden_concat)
        logvar = self.fc_logvar(hidden_concat)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        decoder_input = z.unsqueeze(1).repeat(1, self.pred_len, 1)
        decoder_output, _ = self.decoder_lstm(decoder_input)
        output = self.fc_out(decoder_output).squeeze(-1)
        return output

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

def kld_loss(mu, logvar):
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / mu.size(0)

def train_lstm_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    criterion = nn.MSELoss()

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()

        pred = model(X)
        loss = criterion(pred, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)

    return total_loss / len(loader.dataset)

def eval_lstm(model, loader, device):
    model.eval()
    criterion = nn.MSELoss()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = criterion(pred, y)
            total_loss += loss.item() * X.size(0)

            all_preds.append(pred.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    mse = total_loss / len(loader.dataset)
    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    return mse, all_preds, all_targets

def train_vae_epoch(model, loader, optimizer, device, beta=0.1):
    model.train()
    total_loss = 0.0
    total_recon = 0.0
    total_kld = 0.0
    criterion = nn.MSELoss()

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()

        recon, mu, logvar = model(X)
        recon_loss = criterion(recon, y)
        kld = kld_loss(mu, logvar)
        loss = recon_loss + beta * kld

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)
        total_recon += recon_loss.item() * X.size(0)
        total_kld += kld.item() * X.size(0)

    n = len(loader.dataset)
    return total_loss / n, total_recon / n, total_kld / n

def eval_vae(model, loader, device, num_samples=20):
    model.eval()
    all_samples = []
    all_targets = []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            batch_samples = []

            for _ in range(num_samples):
                recon, _, _ = model(X)
                batch_samples.append(recon.cpu().numpy())

            batch_samples = np.stack(batch_samples, axis=1)
            all_samples.append(batch_samples)
            all_targets.append(y.numpy())

    all_samples = np.concatenate(all_samples, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    return all_samples, all_targets

def compute_detailed_metrics(lstm_preds, vae_samples, targets, sequences, data_handler, vae_num_samples=20):
    lstm_preds_original = []
    vae_samples_original = []
    targets_original = []

    for i in range(len(targets)):
        original_target = sequences[i]['original_output']
        targets_original.append(original_target)

        lstm_pred_original = data_handler.global_scaler.inverse_transform(
            lstm_preds[i].reshape(-1, 1)
        ).flatten()
        lstm_preds_original.append(lstm_pred_original)

        vae_sample_original = []
        for j in range(vae_num_samples):
            sample_original = data_handler.global_scaler.inverse_transform(
                vae_samples[i, j].reshape(-1, 1)
            ).flatten()
            vae_sample_original.append(sample_original)
        vae_samples_original.append(vae_sample_original)

    lstm_preds_original = np.array(lstm_preds_original)
    vae_samples_original = np.array(vae_samples_original)
    targets_original = np.array(targets_original)

    lstm_mse_per_seq = np.mean((lstm_preds_original - targets_original) ** 2, axis=1)

    vae_best_mse_per_seq = []
    vae_best_preds = []
    diversity_per_seq = []

    for i in range(len(targets_original)):
        sample_mses = np.mean((vae_samples_original[i] - targets_original[i]) ** 2, axis=1)
        best_idx = np.argmin(sample_mses)
        vae_best_mse_per_seq.append(sample_mses[best_idx])
        vae_best_preds.append(vae_samples_original[i][best_idx])
        diversity_per_seq.append(np.std(vae_samples_original[i], axis=0).mean())

    vae_best_mse_per_seq = np.array(vae_best_mse_per_seq)
    vae_best_preds = np.array(vae_best_preds)
    diversity_per_seq = np.array(diversity_per_seq)

    improvement = lstm_mse_per_seq - vae_best_mse_per_seq

    lstm_mse_original = np.mean(lstm_mse_per_seq)
    vae_best_mse_original = np.mean(vae_best_mse_per_seq)
    diversity_std = np.mean(diversity_per_seq)

    threshold = np.median(lstm_mse_per_seq)

    coverage_count = 0
    for i in range(len(targets_original)):
        target = targets_original[i]
        samples = vae_samples_original[i]

        covered = True
        for t in range(len(target)):
            min_pred = np.min(samples[:, t])
            max_pred = np.max(samples[:, t])
            if not (min_pred - threshold <= target[t] <= max_pred + threshold):
                covered = False
                break

        if covered:
            coverage_count += 1

    coverage = coverage_count / len(targets_original)

    top_5_indices = np.argsort(improvement)[-5:][::-1]

    top_5_data = []
    for idx in top_5_indices:
        top_5_data.append({
            'idx': idx,
            'lstm_mse': lstm_mse_per_seq[idx],
            'vae_best_mse': vae_best_mse_per_seq[idx],
            'improvement': improvement[idx],
            'y_true': targets_original[idx],
            'y_lstm': lstm_preds_original[idx],
            'y_vae_best': vae_best_preds[idx],
            'diversity_std': diversity_per_seq[idx]
        })

    improvement_buckets = {
        'VAE<<劣(>1000)': (-np.inf, -1000),
        'VAE劣(200~1000)': (-1000, -200),
        'VAE劣(50~200)': (-200, -50),
        'VAE劣(10~50)': (-50, -10),
        'VAE略劣(<10)': (-10, 0),
        '~打平(±10)': (-10, 10),
        'VAE略勝(10~50)': (10, 50),
        'VAE勝(50~200)': (50, 200),
        'VAE大勝(200~1000)': (200, 1000),
        'VAE>>大勝(>1000)': (1000, np.inf)
    }

    bucket_counts = {bucket: 0 for bucket in improvement_buckets.keys()}

    for imp in improvement:
        for bucket, (low, high) in improvement_buckets.items():
            if low <= imp < high:
                bucket_counts[bucket] += 1
                break

    total_seqs = len(improvement)
    bucket_ratios = {bucket: count/total_seqs for bucket, count in bucket_counts.items()}

    return {
        'lstm_mse_original': lstm_mse_original,
        'vae_best_mse_original': vae_best_mse_original,
        'diversity_std': diversity_std,
        'coverage': coverage,
        'lstm_preds_original': lstm_preds_original,
        'vae_samples_original': vae_samples_original,
        'targets_original': targets_original,
        'lstm_mse_per_seq': lstm_mse_per_seq,
        'threshold': threshold,
        'top_5_data': top_5_data,
        'bucket_counts': bucket_counts,
        'bucket_ratios': bucket_ratios,
        'total_seqs': total_seqs
    }

def print_top_5_table(top_5_data):
    print("\nTop-5 Regressed (VAE best >> LSTM) ===")
    print(f"{'idx':<5} {'LSTM_MSE':<12} {'VAE_best_MSE':<15} {'Δ(LSTM-VAE)':<15} {'y_true':<20} {'y_LSTM':<45} {'y_VAE_best':<45} {'Diversity_std':<15}")
    print("-" * 180)

    for i, data in enumerate(top_5_data):
        print(f"{i:<5} {data['lstm_mse']:<12.2f} {data['vae_best_mse']:<15.2f} {data['improvement']:<15.2f} "
              f"{str([round(x, 1) for x in data['y_true'].tolist()]):<20} {str([round(x, 1) for x in data['y_lstm'].tolist()]):<45} "
              f"{str([round(x, 1) for x in data['y_vae_best'].tolist()]):<45} {data['diversity_std']:<15.6f}")

def print_improvement_buckets(bucket_counts, bucket_ratios, total_seqs):
    print(f"\n# Min-rate by improvement bucket (Δ = LSTM MSE - VAE best MSE)")
    print(f"{'Improvement bucket':<25} {'count':<10} {'ratio':<10}")
    print("-" * 50)

    buckets = [
        'VAE<<劣(>1000)', 'VAE劣(200~1000)', 'VAE劣(50~200)', 'VAE劣(10~50)', 'VAE略劣(<10)',
        '~打平(±10)', 'VAE略勝(10~50)', 'VAE勝(50~200)', 'VAE大勝(200~1000)', 'VAE>>大勝(>1000)'
    ]

    for bucket in buckets:
        count = bucket_counts[bucket]
        ratio = bucket_ratios[bucket]
        print(f"{bucket:<25} {count:<10} {ratio:<10.4f}")

def plot_comparison_original(metrics, example_idx=0):
    lstm_preds = metrics['lstm_preds_original']
    vae_samples = metrics['vae_samples_original']
    targets = metrics['targets_original']

    plt.figure(figsize=(12, 6))

    future_weeks = np.arange(1, len(targets[example_idx]) + 1)

    for i in range(min(20, vae_samples.shape[1])):
        plt.plot(future_weeks, vae_samples[example_idx, i], 'b-', alpha=0.1, linewidth=1)

    plt.plot(future_weeks, lstm_preds[example_idx], 'g-', linewidth=2, label='LSTM Prediction')
    plt.plot(future_weeks, targets[example_idx], 'r-', linewidth=3, label='Ground Truth')

    plt.xlabel('Future Weeks')
    plt.ylabel('Clicks (Original Scale)')
    plt.title(f'Sequence Prediction Comparison - Example {example_idx} (Original Scale)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def main_flow(seq_len=4,
              pred_len=2,
              batch_size=128,
              lstm_hidden=128,
              vae_enc_hidden=128,
              vae_latent=32,
              epochs=30,
              beta=0.1,
              vae_num_samples=20):

    # 檔案上傳（只在需要時上傳）
    print("=== 檔案上傳檢查 ===")
    upload_files_if_needed()

    dh = DataHandler(min_weeks_needed=12)

    student_info, student_vle, student_assessment = dh.load_oulad_or_synthetic()

    weekly = dh.preprocess_weekly(student_info, student_vle, student_assessment)
    print(f"📈 Weekly data shape: {weekly.shape}")
    print(f"👥 Unique students: {weekly['student_id'].nunique()}")

    sequences = dh.create_sequences(weekly, seq_length=seq_len, pred_length=pred_len)

    student_ids = list(set([s['student_id'] for s in sequences]))
    train_ids, temp_ids = train_test_split(student_ids, test_size=0.3, random_state=SEED)
    val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=SEED)

    train_seq = [s for s in sequences if s['student_id'] in train_ids]
    val_seq = [s for s in sequences if s['student_id'] in val_ids]
    test_seq = [s for s in sequences if s['student_id'] in test_ids]

    print(f"📚 Train sequences: {len(train_seq)}")
    print(f"📊 Val sequences: {len(val_seq)}")
    print(f"🧪 Test sequences: {len(test_seq)}")

    train_loader = DataLoader(SequenceDataset(train_seq), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(SequenceDataset(val_seq), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(SequenceDataset(test_seq), batch_size=batch_size, shuffle=False)

    print("\n=== Training Seq2Seq LSTM ===")
    lstm_model = Seq2SeqLSTM(
        input_dim=4,
        enc_hidden=lstm_hidden,
        dec_hidden=lstm_hidden,
        num_layers=2,
        pred_len=pred_len
    ).to(device)

    lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=1e-3)

    lstm_train_losses = []
    lstm_val_losses = []

    for epoch in range(epochs):
        train_loss = train_lstm_epoch(lstm_model, train_loader, lstm_optimizer, device)
        val_mse, _, _ = eval_lstm(lstm_model, val_loader, device)

        lstm_train_losses.append(train_loss)
        lstm_val_losses.append(val_mse)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val MSE: {val_mse:.4f}")

    lstm_test_mse, lstm_preds, lstm_targets = eval_lstm(lstm_model, test_loader, device)
    print(f"\nLSTM Test MSE (normalized): {lstm_test_mse:.4f}")

    print("\n=== Training Seq2Seq VAE ===")
    vae_model = Seq2SeqVAE(
        input_dim=4,
        enc_hidden=vae_enc_hidden,
        latent_dim=vae_latent,
        dec_hidden=vae_enc_hidden,
        num_layers=2,
        pred_len=pred_len
    ).to(device)

    vae_optimizer = optim.Adam(vae_model.parameters(), lr=1e-3)

    vae_train_losses = []
    vae_val_losses = []

    for epoch in range(epochs):
        train_loss, recon_loss, kld_loss = train_vae_epoch(vae_model, train_loader, vae_optimizer, device, beta=beta)

        vae_model.eval()
        val_recon = 0.0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                recon, _, _ = vae_model(X)
                val_recon += nn.MSELoss()(recon, y).item() * X.size(0)

        val_loss = val_recon / len(val_loader.dataset)

        vae_train_losses.append(train_loss)
        vae_val_losses.append(val_loss)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} (Recon: {recon_loss:.4f}, KLD: {kld_loss:.4f}), Val Loss: {val_loss:.4f}")

    vae_samples, vae_targets = eval_vae(vae_model, test_loader, device, num_samples=vae_num_samples)

    metrics = compute_detailed_metrics(
        lstm_preds, vae_samples, vae_targets, test_seq, dh, vae_num_samples
    )

    print(f"\n=== Evaluation Results (Original Scale) ===")
    print(f"LSTM MSE: {metrics['lstm_mse_original']:.4f}")
    print(f"VAE Best-of-{vae_num_samples} MSE: {metrics['vae_best_mse_original']:.4f}")
    print(f"VAE Diversity (std): {metrics['diversity_std']:.4f}")
    print(f"VAE Coverage: {metrics['coverage']:.4f}")
    print(f"(threshold τ = LSTM per-sequence MSE median = {metrics['threshold']:.4f})")

    print_top_5_table(metrics['top_5_data'])
    print_improvement_buckets(metrics['bucket_counts'], metrics['bucket_ratios'], metrics['total_seqs'])

    print("\n=== Visualizing Results (Original Scale) ===")
    for i in range(min(3, len(metrics['targets_original']))):
        plot_comparison_original(metrics, example_idx=i)

    torch.save(lstm_model.state_dict(), "seq2seq_lstm.pth")
    torch.save(vae_model.state_dict(), "seq2seq_vae.pth")
    print("\n💾 Models saved successfully!")

    return metrics

if __name__ == "__main__":
    results = main_flow(
        seq_len=4,
        pred_len=2,
        batch_size=128,
        lstm_hidden=128,
        vae_enc_hidden=128,
        vae_latent=32,
        epochs=30,
        beta=0.1,
        vae_num_samples=20
    )

    print("\n=== Final Results (Original Scale) ===")
    print(f"LSTM MSE: {results['lstm_mse_original']:.4f}")
    print(f"VAE Best-of-N MSE: {results['vae_best_mse_original']:.4f}")
    print(f"VAE Diversity: {results['diversity_std']:.4f}")
    print(f"VAE Coverage: {results['coverage']:.4f}")

Device: cpu
=== 檔案上傳檢查 ===
📁 請上傳 OULAD 資料集檔案
   - studentInfo.csv
   - studentVle.csv
   - studentAssessment.csv
   或上傳包含這些檔案的 ZIP 壓縮檔
